In [ ]:
%pip install --quiet -U langchain_openai langchain_core langchain_community langchain-tavily
%pip install --quiet -U langchain_openai langchain_core langchain_community langchain-tavily
%pip install truststore

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from dotenv import load_dotenv
load_dotenv()
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")
_set_env("LANGSMITH_API_KEY")
print("API KEY EXISTS:", bool(os.environ.get("LANGSMITH_API_KEY")))
print("ENDPOINT:", os.environ.get("LANGSMITH_ENDPOINT"))
print("TRACING:", os.environ.get("LANGSMITH_TRACING"))
print("PROJECT:", os.environ.get("LANGSMITH_PROJECT"))

API KEY EXISTS: True
ENDPOINT: https://api.smith.langchain.com
TRACING: true
PROJECT: langchain-academy


In [ ]:
from langchain_openai import ChatOpenAI
model1 = ChatOpenAI(model_name="gpt-4.1-mini", temperature=0.7, max_tokens=1000)
model2 = ChatOpenAI(model_name="gpt-3.5-turbo-0125", temperature=0.7, max_tokens=1000)

Two methods invoke and stream
LangGraph Usage

Inside a LangGraph node, you'll usually see:

def chatbot(state):
    return {
        "messages": [
            llm.invoke(state["messages"])
        ]
    }

because the graph needs the complete message before moving to the next node.

Streaming is typically enabled when running the graph:

for event in graph.stream(
    {"messages": [("user", "Hi")]},
    stream_mode="values"
):
    print(event)

Here:

Graph execution streams updates.
Internally, nodes may still use invoke().

So there are two different streaming concepts:

LLM streaming
llm.stream(...)

Streams model tokens.

Graph streaming
graph.stream(...)

Streams state updates/events from the graph.

As you learn LangGraph, keep these separate:

ChatOpenAI.stream()
    => token streaming

Graph.stream()
    => workflow/state streaming

That's one of the most important distinctions in LangGraph.


for chunk in llm.stream("Explain LangGraph"):
    print(chunk.content, end="")

the for loop does not collect all chunks first.

What happens is:

Request goes to the LLM.
First token/chunk arrives.
Loop executes one iteration.
Loop waits for the next chunk.
Next chunk arrives.
Loop executes again.
This continues until the model signals end of stream.

Conceptually:

Chunk 1 arrives  ---> loop body runs
(wait)

Chunk 2 arrives  ---> loop body runs
(wait)

Chunk 3 arrives  ---> loop body runs
(wait)

End of stream    ---> loop exits

It's similar to:

while True:
    chunk = get_next_chunk()

    if chunk is END:
        break

    process(chunk)

So the loop waits when there is no next token yet. It doesn't break because of a temporary size limit. It breaks only when the streaming generator signals that the response is finished.

For example, if the model generates:

LangGraph is a framework for AI agents

the chunks might arrive as:

"Lang"
"Graph"
" is"
" a"
" framework"
" for"
" AI"
" agents"

and the for loop processes each chunk as it arrives, waiting between chunks.

Messages have a role (that describes who is saying the message) and a content property

In [ ]:
import os
import truststore
truststore.inject_into_ssl()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

model1 = ChatOpenAI(model="gpt-4o-mini")
result = model1.invoke([HumanMessage(content="What is the capital of France?")])
print(result.content)

ModuleNotFoundError: No module named 'truststore'